<a href="https://colab.research.google.com/github/mdaminu2002-sketch/bank_fraud/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-deps unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 41.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00


In [ ]:
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install sentencepiece protobuf datasets huggingface_hub
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# ----------------------------------------------------
# A. Get 2,000 Hausa Pairs & Format Dataset
# ----------------------------------------------------
print("1/3 Downloading and formatting Hausa dataset...")
raw_dataset = load_dataset("michsethowusu/english-hausa_sentence-pairs_mt560", split="train[:2000]")

formatted_data = []
for item in raw_dataset:
    en_text = item.get("english", "")
    ha_text = item.get("hausa", "")
    if en_text and ha_text:
        formatted_data.append({
            "messages": [
                {"role": "user", "content": f"Translate to Hausa or respond in Hausa:\n{en_text}"},
                {"role": "model", "content": ha_text}
            ]
        })

# Save locally inside Colab
with open("hausa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print("Saved hausa_dataset.json inside Colab!")

# ----------------------------------------------------
# B. Load Base Gemma Model (4-bit QLoRA for Fast VRAM)
# ----------------------------------------------------
print("2/3 Loading Gemma model into Colab GPU...")
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it", # Extremely fast & lightweight for free Colab GPU
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# ----------------------------------------------------
# C. Add LoRA Adapters
# ----------------------------------------------------
print("3/3 Setting up LoRA training adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

print("Setup Complete! Ready for Fine-Tuning.")

In [ ]:
from datasets import load_dataset
import json

# Download open QA datasets for science, math, and humanities
academic_ds = load_dataset("allenai/sciq", split="train[:15000]")

formatted_dataset = []

for row in academic_ds:
    question = row["question"]
    answer = row["correct_answer"] + ". " + row["support"]

    formatted_dataset.append({
        "messages": [
            {"role": "user", "content": f"Academic Question: {question}"},
            {"role": "model", "content": answer}
        ]
    })

# Save base academic questions
with open("academic_pairs.json", "w", encoding="utf-8") as f:
    json.dump(formatted_dataset, f, ensure_ascii=False, indent=2)

print(f"Loaded {len(formatted_dataset)} academic QA pairs!")

In [ ]:
import json

# 1. Load files
with open("hausa_dataset.json", "r", encoding="utf-8") as f:
    hausa_data = json.load(f)

with open("academic_pairs.json", "r", encoding="utf-8") as f:
    academic_data = json.load(f)

# 2. Combine into master list
master_dataset = hausa_data + academic_data

# 3. Save as the final training dataset
with open("abu_master_dataset.json", "w", encoding="utf-8") as f:
    json.dump(master_dataset, f, ensure_ascii=False, indent=2)

print(f"Total entries ready for fine-tuning: {len(master_dataset)}")

In [ ]:
%pip install ollama

In [ ]:
from datasets import Dataset, Features, Image, Value

def format_abu_master_entry(row):
    """
    Formats raw ABU master data into standard multi-modal chat format.
    """
    # If the entry includes an image (e.g., ID card or user upload)
    if row.get("image"):
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": row["image"]},
                        {"type": "text", "text": row["instruction_prompt"]}
                    ]
                },
                {
                    "role": "assistant",
                    "content": row["expected_response"]
                }
            ]
        }
    else:
        # Standard text-only entry for chat
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": row["instruction_prompt"]}]
                },
                {
                    "role": "assistant",
                    "content": row["expected_response"]
                }
            ]
        }

# Example: Adding to your master dataset script
# abu_dataset = abu_dataset.map(format_abu_master_entry)

In [ ]:
# import base64
# from fastapi import FastAPI, UploadFile, File, Form
# from fastapi.middleware.cors import CORSMiddleware
# from pydantic import BaseModel
# import ollama

# app = FastAPI()

# # Enable CORS for your frontend to communicate freely
# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=["*"],
#     allow_credentials=True,
#     allow_methods=["*"],
#     allow_headers=["*"],
# )

# MODEL_NAME = "abu-study-buddy"

# # -------------------------------------------------------------------
# # ROUTE 1: Chat Box (Hausa + General Knowledge)
# # -------------------------------------------------------------------
# class ChatRequest(BaseModel):
#     user_message: str

# @app.post("/api/chat")
# async def chat_endpoint(request: ChatRequest):
#     """
#     Handles regular chat requests for Hausa tutoring, academic Q&A, and campus info.
#     """
#     response = ollama.chat(
#         model=MODEL_NAME,
#         messages=[
#             {
#                 "role": "system",
#                 "content": "You are the ABU Zaria Study Buddy AI. Respond naturally in English or Hausa."
#             },
#             {
#                 "role": "user",
#                 "content": request.user_message
#             }
#         ]
#     )
#     return {"reply": response["message"]["content"]}


# # -------------------------------------------------------------------
# # ROUTE 2: Registration & Document Verification (Vision API)
# # -------------------------------------------------------------------
# @app.post("/api/verify-document")
# async def verify_document(
#     document_type: str = Form(...),  # e.g., "Birth Certificate", "State of Origin", "JAMB Slip"
#     file: UploadFile = File(...)
# ):
#     """
#     Receives an image uploaded by a student, feeds it into Ollama's vision pipeline,
#     and returns document validity analysis.
#     """
#     # 1. Read the image file uploaded from the frontend
#     image_bytes = await file.read()

#     # 2. Encode to base64 for Ollama Vision API
#     base64_image = base64.b64encode(image_bytes).decode('utf-8')

#     # 3. Create a strict verification prompt
#     prompt = f"""
#     Analyze this uploaded document.
#     The student claims this is a: '{document_type}'.

#     Tasks:
#     1. Check if the document in the image is indeed a valid {document_type}.
#     2. Extract key details (Name, Registration/ID Number, Issue Date, State/LGA if visible).
#     3. If it is NOT a {document_type}, identify what document it actually is and explain what went wrong.
#     """

#     response = ollama.chat(
#         model=MODEL_NAME,
#         messages=[
#             {
#                 "role": "user",
#                 "content": prompt,
#                 "images": [base64_image]  # Send base64 image data to vision model
#             }
#         ]
#     )

#     return {
#         "analysis": response["message"]["content"]
#     }

In [ ]:
# Install Unsloth and training dependencies
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install sentencepiece protobuf datasets huggingface_hub
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from google.colab import files

# 1. Load Base Gemma / Llama Model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Load Your Master ABU Dataset
dataset = load_dataset("json", data_files="abu_master_dataset.json", split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        prompt = f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n<|im_start|>assistant\n{messages[1]['content']}<|im_end|>"
        texts.append(prompt)
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

# 4. Train Model
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # Set num_train_epochs = 3 for complete full pass
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "abu_study_buddy_outputs",
    ),
)

trainer.train()

# 5. Export and Download GGUF to Your PC
#model.save_pretrained_gguf("abu_study_buddy", tokenizer, quantization_method = "q4_k_m")
#files.download("abu_study_buddy/abu_study_buddy-Q4_K_M.gguf")

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# 1. Load the formatted dataset
dataset = load_dataset("json", data_files="abu_master_dataset.json", split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        prompt = f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n<|im_start|>assistant\n{messages[1]['content']}<|im_end|>"
        texts.append(prompt)
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

# 2. Configure Training Arguments using SFTConfig
training_args = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, # <--- Changed from 2 to 1 to fix PicklingError!
    packing = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 120,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    output_dir = "abu_study_buddy_outputs",
)

# 3. Create Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    args = training_args,
)

# 4. Start Training
print("Starting Fine-Tuning...")
trainer.train()
print("Training Finished Successfully!")

In [ ]:
from google.colab import files

# 1. Convert fine-tuned LoRA weights into a standalone 4-bit GGUF file
model.save_pretrained_gguf("abu_study_buddy_model", tokenizer, quantization_method = "q4_k_m")

# 2. Trigger automatic browser download to your PC's Downloads folder
files.download("abu_study_buddy_model/abu_study_buddy_model-Q4_K_M.gguf")

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# 1. Enable fast native inference mode in Unsloth (2x faster)
FastLanguageModel.for_inference(model)

# 2. Define a test function to prompt your model
def test_abu_study_buddy(prompt_text):
    print(f"\n--- USER QUESTION: {prompt_text} ---")

    # Format prompt using the same ChatML format used during training
    formatted_prompt = f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"

    # Tokenize input and transfer to GPU
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

    # Set up real-time text streaming output
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)

    # Generate model response
    _ = model.generate(
        **inputs,
        streamer = text_streamer,
        max_new_tokens = 256,
        use_cache = True,
        temperature = 0.5, # Lower temperature = more precise academic answers
    )

# ----------------------------------------------------
# 3. RUN TESTS ACROSS ALL YOUR FINE-TUNING CATEGORIES
# ----------------------------------------------------

# Test A: Hausa Knowledge
test_abu_study_buddy("Bani bayanin Second Law of Thermodynamics a Hausa.")

# Test B: ABU Campus Location / Registration
test_abu_study_buddy("Where is the Faculty of Law located in ABU Zaria?")

# Test C: Final Year Clearance
test_abu_study_buddy("What documents do I need for final year clearance at KIL Library?")

# Test D: IT / SIWES Placement
test_abu_study_buddy("I am a 300L Computer Science student in ABU. Where should I apply for my IT placement?")